In [2]:
import os
import gc
import sys
import glob
import numpy as np
import pandas as pd
import netCDF4 as nc
from datetime import datetime, timedelta

In [3]:
# To use PLUMBER2_GPP_common_utils, change directory to where it exists
os.chdir('/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2')
from PLUMBER2_GPP_common_utils import *

In [4]:
# Path of PLUMBER 2 dataset
PLUMBER2_path      = "/srv/ccrc/LandAP/z5218916/data/PLUMBER2/"
PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"
PLUMBER2_met_path  = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Met/"

site_names, IGBP_types, clim_types, model_names = load_default_list()

remove_site        = get_removed_site_names()

models_calc_LAI   = ['ORC2_r6593','ORC2_r6593_CO2','ORC3_r7245_NEE','ORC3_r8120','GFDL','SDGVM','QUINCY','NoahMPv401']
model_LAI_names   = {'ORC2_r6593':'lai','ORC2_r6593_CO2':'lai','ORC3_r7245_NEE':'lai','ORC3_r8120':'lai',
                     'GFDL':'lai', 'SDGVM':'lai','QUINCY':'LAI','NoahMPv401':'LAI'} #

# Calculate remaining sites
set_site_names      = set(site_names)
set_remove_site     = set(remove_site)
remain_sites        = set_site_names - set_remove_site
remain_sites        = list(remain_sites)

<h5 style="color:orange;">Check time interval</h5>  

In [4]:
for i, site_name in enumerate(remain_sites): 
    # Define paths
    PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
    PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"

    # Find the correct file
    file_path = glob.glob(PLUMBER2_flux_path + f"*{site_name}*.nc")

    if not file_path:
        print(f"No file found for {site_name}")
        continue  # Skip to the next site

    # Initialize a dictionary to store time differences
    time_diff = {}

    # Open the NetCDF file
    with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
        # Find variables containing 'time' in their name
        time_vars = [var for var in f.variables if 'time' in var.lower()]

        # Loop through the time variables and calculate time differences
        for time_var in time_vars: 
            time_tmp = nc.num2date(f.variables[time_var][:], f.variables[time_var].units,
                                   only_use_cftime_datetimes=False, only_use_python_datetimes=True)

            if len(time_tmp) > 1:
                time_diff[time_var] = (time_tmp[1] - time_tmp[0]).seconds
            else:
                time_diff[time_var] = None  # Not enough time steps to calculate difference

        # Print the time differences for this site
        print(f"Site: {site_name}, Time Differences: {time_diff}")

Site: US-KS2, Time Differences: {'CABLE_time': 1800, 'CABLE-POP-CN_time': 1800, 'CHTESSEL_Ref_exp1_time': 1800, 'CLM5a_time': 1800, 'GFDL_time': 1800, 'JULES_GL9_withLAI_time': 1800, 'MATSIRO_time': 1800, 'MuSICA_time': 1800, 'NoahMPv401_time': 1800, 'ORC2_r6593_time': 1800, 'ORC3_r8120_time': 1800, 'QUINCY_time': 1800, '1lin_time': 1800, '3km27_time': 1800, '6km729_time': 1800, '6km729lag_time': 1800, 'RF_eb_time': 1800, 'RF_raw_time': 1800, 'LSTM_eb_time': 1800, 'LSTM_raw_time': 1800, 'JULES_GL9_time': 1800, 'NASAEnt_time': 1800, 'STEMMUS-SCOPE_time': 1800}
Site: US-Cop, Time Differences: {'CABLE_time': 3600, 'CABLE-POP-CN_time': 3600, 'CHTESSEL_Ref_exp1_time': 3600, 'CLM5a_time': 3600, 'GFDL_time': 3600, 'JULES_GL9_withLAI_time': 3600, 'MATSIRO_time': 3600, 'MuSICA_time': 3600, 'NoahMPv401_time': 3600, 'ORC2_r6593_time': 3600, 'ORC3_r8120_time': 3600, 'QUINCY_time': 3600, '1lin_time': 3600, '3km27_time': 3600, '6km729_time': 3600, '6km729lag_time': 3600, 'RF_eb_time': 3600, 'RF_raw_

<h5 style="color:orange;">Check total time steps</h5>  

In [5]:
for i, site_name in enumerate(remain_sites): 
    
    # Define paths
    PLUMBER2_path_site = f"/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_GPP_PLUMBER2/nc_files/{site_name}.nc"
    PLUMBER2_flux_path = "/srv/ccrc/LandAP/z5218916/data/Fluxnet_data/Post-processed_PLUMBER2_outputs/Nc_files/Flux/"

    # Find the correct file
    file_path = glob.glob(PLUMBER2_flux_path + f"*{site_name}*.nc")

    if not file_path:
        print(f"No file found for {site_name}")
        continue  # Skip to the next site

    # Initialize a dictionary to store time differences
    time_length = {}

    # Open the NetCDF file
    with nc.Dataset(PLUMBER2_path_site, mode='r') as f:
        # Find variables containing 'time' in their name
        time_vars = [var for var in f.variables if 'time' in var.lower()]

        # Loop through the time variables and calculate time differences
        for time_var in time_vars: 
            time_tmp = nc.num2date(f.variables[time_var][:], f.variables[time_var].units,
                                   only_use_cftime_datetimes=False, only_use_python_datetimes=True)
            time_length[time_var] = len(time_tmp)
            
        if len(set(time_length.values())) != 1:
            # Print the time differences for this site
            print(f"Site: {site_name}, Time Differences: {time_length}")

Site: US-Ha1, Time Differences: {'CABLE_time': 184104, 'CABLE-POP-CN_time': 184104, 'CHTESSEL_Ref_exp1_time': 184104, 'CLM5a_time': 184104, 'GFDL_time': 184104, 'JULES_GL9_withLAI_time': 184104, 'MATSIRO_time': 184104, 'MuSICA_time': 184104, 'NoahMPv401_time': 184104, 'ORC2_r6593_time': 184104, 'ORC3_r8120_time': 184104, 'QUINCY_time': 192864, '1lin_time': 184104, '3km27_time': 184104, '6km729_time': 184104, '6km729lag_time': 184104, 'RF_eb_time': 184104, 'RF_raw_time': 184104, 'LSTM_eb_time': 184104, 'LSTM_raw_time': 184104, 'JULES_GL9_time': 184104, 'NASAEnt_time': 184104, 'STEMMUS-SCOPE_time': 184104}
